In [2]:
import cv2
import numpy as np
import math

# Load image
# img = cv2.imread("diagram.png")
img = cv2.imread("../../Graph Generation/Ground_truth/PDF/feature.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Threshold to binary
_, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

# Optional: clean noise
kernel = np.ones((3,3), np.uint8)
clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)


In [3]:
contours, _ = cv2.findContours(clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

rectangles = []
for cnt in contours:
    approx = cv2.approxPolyDP(cnt, 0.02*cv2.arcLength(cnt, True), True)
    if len(approx) == 4:  # quadrilateral
        x,y,w,h = cv2.boundingRect(approx)
        rectangles.append((x,y,w,h))

# Features
areas = [w*h for (_,_,w,h) in rectangles]
rect_count = len(rectangles)
rect_coverage = sum(areas) / (img.shape[0]*img.shape[1])
avg_rect_area = np.mean(areas) if areas else 0
rect_stdev = np.std(areas) if areas else 0


In [4]:
edges = cv2.Canny(clean, 50, 150, apertureSize=3)
lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=30, maxLineGap=5)

line_lengths = []
line_angles = []
if lines is not None:
    for l in lines:
        x1,y1,x2,y2 = l[0]
        length = math.hypot(x2-x1, y2-y1)
        angle = math.degrees(math.atan2(y2-y1, x2-x1))
        line_lengths.append(length)
        line_angles.append(angle)

# Features
avg_line_length = np.mean(line_lengths) if line_lengths else 0
longest_line = np.max(line_lengths) if line_lengths else 0
shortest_line = np.min(line_lengths) if line_lengths else 0
line_length_stdev = np.std(line_lengths) if line_lengths else 0
avg_line_angle = np.mean(line_angles) if line_angles else 0


In [5]:
# Orthogonality: ratio of lines near 0°/90°/180°
orth_count = sum(1 for a in line_angles if abs(a % 90) < 5)
orth_ratio = orth_count / len(line_angles) if line_angles else 0

# Crossings: check intersections between line segments
def intersect(l1, l2):
    # simple segment intersection test
    (x1,y1,x2,y2) = l1
    (x3,y3,x4,y4) = l2
    # vector cross product approach
    def ccw(A,B,C): return (C[1]-A[1])*(B[0]-A[0]) > (B[1]-A[1])*(C[0]-A[0])
    return ccw((x1,y1),(x3,y3),(x4,y4)) != ccw((x2,y2),(x3,y3),(x4,y4)) and \
           ccw((x1,y1),(x2,y2),(x3,y3)) != ccw((x1,y1),(x2,y2),(x4,y4))

crossings = 0
if lines is not None:
    segs = [tuple(l[0]) for l in lines]
    for i in range(len(segs)):
        for j in range(i+1, len(segs)):
            if intersect(segs[i], segs[j]):
                crossings += 1


In [14]:
bends = []
for cnt in contours:
    approx = cv2.approxPolyDP(cnt, 0.02*cv2.arcLength(cnt, True), True)
    if len(approx) > 2:
        bends.append(len(approx))  # number of bends
avg_line_bends = np.mean(bends) if bends else 0


In [15]:
crossing_angles = []
for i in range(len(segs)):
    for j in range(i+1, len(segs)):
        if intersect(segs[i], segs[j]):
            (x1,y1,x2,y2) = segs[i]
            (x3,y3,x4,y4) = segs[j]
            a1 = math.atan2(y2-y1, x2-x1)
            a2 = math.atan2(y4-y3, x4-x3)
            crossing_angles.append(abs(a1-a2))
avg_crossing_angle = np.mean(crossing_angles) if crossing_angles else 0


In [16]:
dists = []
for i in range(len(rectangles)):
    for j in range(i+1, len(rectangles)):
        (x1,y1,w1,h1) = rectangles[i]
        (x2,y2,w2,h2) = rectangles[j]
        # center points
        c1 = (x1+w1/2, y1+h1/2)
        c2 = (x2+w2/2, y2+h2/2)
        dists.append(math.hypot(c2[0]-c1[0], c2[1]-c1[1]))
avg_shortest_distance = np.mean(dists) if dists else 0


In [17]:
centers = [(x+w/2, y+h/2) for (x,y,w,h) in rectangles]
if centers:
    cx = [c[0] for c in centers]
    cy = [c[1] for c in centers]
    rect_distribution = np.var(cx) + np.var(cy)
else:
    rect_distribution = 0


In [18]:
orth_count = 0
for (x,y,w,h) in rectangles:
    aspect = w/h if h>0 else 0
    if abs(aspect-1) < 0.1:  # nearly square
        orth_count += 1
rect_orth = orth_count / rect_count if rect_count else 0

# RectOrth2: maybe alignment of rectangle centers along grid lines
if centers:
    mean_x = np.mean([c[0] for c in centers])
    aligned = sum(1 for c in centers if abs(c[0]-mean_x) < 5)
    rect_orth2 = aligned / rect_count
else:
    rect_orth2 = 0


In [19]:
features = {
    "RectCoverage": rect_coverage,
    "AvgRectArea": avg_rect_area,
    "RectStDev": rect_stdev,
    "AspectRatio": np.mean([w/h for (_,_,w,h) in rectangles if h>0]) if rectangles else 0,
    "AvgLineLength": avg_line_length,
    "LongestLine": longest_line,
    "ShortestLine": shortest_line,
    "LineLengthStDev": line_length_stdev,
    "AvgLineAngle": avg_line_angle,
    "OrthLinesRatio": orth_ratio,
    "LineCrossings": crossings,
    "rectangles": rect_count,
    "lines": len(lines) if lines is not None else 0,
    "AvgLineBends": avg_line_bends,
    "AvgCrossingAngle": avg_crossing_angle,
    "AvgShortestDistance": avg_shortest_distance,
    "RectDistribution": rect_distribution,
    "RectOrth": rect_orth,
    "RectOrth2": rect_orth2
}


In [20]:
print(features)

{'RectCoverage': 0.04022821983416139, 'AvgRectArea': np.float64(176.46728971962617), 'RectStDev': np.float64(1147.0314576409344), 'AspectRatio': np.float64(0.7230878556461535), 'AvgLineLength': np.float64(67.70045240444719), 'LongestLine': np.float64(128.0), 'ShortestLine': np.float64(30.0), 'LineLengthStDev': np.float64(29.927913295670663), 'AvgLineAngle': np.float64(-33.854531277468645), 'OrthLinesRatio': 0.8125, 'LineCrossings': 2, 'rectangles': 107, 'lines': 32, 'AvgLineBends': np.float64(4.948148148148148), 'AvgCrossingAngle': np.float64(1.5707963267948966), 'AvgShortestDistance': np.float64(401.94283151882286), 'RectDistribution': np.float64(106199.22246484409), 'RectOrth': 0.48598130841121495, 'RectOrth2': 0.009345794392523364}


In [8]:
import joblib

# Load the trained model (replace with your best one)
model = joblib.load("./best_model_extratrees.pkl")

In [25]:
import pandas as pd

# Wrap features into a DataFrame with one row
X = pd.DataFrame([features])

train_features = model.feature_names_in_
print(train_features)
X = X[train_features]

['RectCoverage' 'AvgRectArea' 'RectStDev' 'AspectRatio' 'AvgLineBends'
 'AvgLineLength' 'LongestLine' 'ShortestLine' 'LineLengthStDev'
 'AvgLineAngle' 'OrthLinesRatio' 'LineCrossings' 'AvgCrossingAngle'
 'AvgShortestDistance' 'RectDistribution' 'RectOrth' 'RectOrth2'
 'rectangles' 'lines']


In [26]:
y_pred = model.predict(X)
print("Predicted score:", y_pred[0])


Predicted score: 3.0611111111111113


In [24]:
print(model.feature_names_in_)


['RectCoverage' 'AvgRectArea' 'RectStDev' 'AspectRatio' 'AvgLineBends'
 'AvgLineLength' 'LongestLine' 'ShortestLine' 'LineLengthStDev'
 'AvgLineAngle' 'OrthLinesRatio' 'LineCrossings' 'AvgCrossingAngle'
 'AvgShortestDistance' 'RectDistribution' 'RectOrth' 'RectOrth2'
 'rectangles' 'lines']
